# Feature Engineering

Reads `data/processed/demand_clean.csv` (from `02_data_cleaning.ipynb`) and builds calendar, lag, rolling, weather-interaction, and cyclical features, saving the result to `data/processed/features.csv` for the baseline, statistical, ML, and DL notebooks to consume. Lag and rolling features are computed on shifted (past-only) values to avoid leakage (proposal.txt Section 9.11).

In [ ]:
import os
import sys
from pathlib import Path

import yaml

# Works both in the local repo and on Kaggle. Locally, config.yaml is read
# from disk. On Kaggle there is no repo checkout -- only whatever cells you
# paste -- so the config is built inline instead. This notebook only reads
# from /kaggle/working (produced by 02_data_cleaning.ipynb earlier in the
# same Kaggle session), never from /kaggle/input, so IS_KAGGLE just needs
# KAGGLE_KERNEL_RUN_TYPE (always set on Kaggle) -- no dataset needs to be
# attached for this notebook specifically.
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    REPO_ROOT = Path("/kaggle/working")
    config = {
        "data": {
            "processed_file": "/kaggle/working/data/processed/demand_clean.csv",
            "features_file": "/kaggle/working/data/processed/features.csv",
            "timestamp_col": "timestamp",
            "target_col": "demand",
            "frequency": "h",
        },
        "features": {
            "lags": [1, 2, 6, 24, 48, 168],
            "rolling_windows": [3, 6, 12, 24, 168],
            "cyclical": ["hour", "day_of_week", "month"],
        },
        "peak_demand": {"percentile_threshold": 0.95, "seasonal": True},
    }
else:
    def find_repo_root(start: Path) -> Path:
        for parent in [start, *start.parents]:
            if (parent / "config.yaml").exists():
                return parent
        raise FileNotFoundError("config.yaml not found in any parent directory")

    REPO_ROOT = find_repo_root(Path.cwd())
    sys.path.insert(0, str(REPO_ROOT))

    with open(REPO_ROOT / "config.yaml") as f:
        config = yaml.safe_load(f)

config

In [12]:
import pandas as pd

# On Kaggle, src/ isn't on disk (only pasted cells are) -- these are
# identical inline copies of src/data/load_data.py and src/features/build_features.py.
# Keep them in sync with src/ if that logic ever changes.
if IS_KAGGLE:
    import numpy as np

    def load_raw_data(path, timestamp_col="timestamp"):
        df = pd.read_csv(path, parse_dates=[timestamp_col])
        return df.sort_values(timestamp_col).reset_index(drop=True)

    def set_timestamp_index(df, timestamp_col="timestamp"):
        df = df.set_index(timestamp_col)
        return df.sort_index()

    def add_calendar_features(df):
        df = df.copy()
        df["hour"] = df.index.hour
        df["day_of_week"] = df.index.dayofweek
        df["month"] = df.index.month
        df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
        return df

    def add_lag_features(df, target_col, lags):
        df = df.copy()
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)
        return df

    def add_rolling_features(df, target_col, windows):
        df = df.copy()
        shifted = df[target_col].shift(1)
        for window in windows:
            df[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
            df[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()
        return df

    def add_cyclical_features(df, col, period):
        df = df.copy()
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)
        return df
else:
    from src.data.load_data import load_raw_data, set_timestamp_index
    from src.features.build_features import (
        add_calendar_features,
        add_cyclical_features,
        add_lag_features,
        add_rolling_features,
    )

PROCESSED_PATH = REPO_ROOT / config["data"]["processed_file"]
FEATURES_PATH = REPO_ROOT / config["data"]["features_file"]
TIMESTAMP_COL = config["data"]["timestamp_col"]
TARGET_COL = config["data"]["target_col"]
LAGS = config["features"]["lags"]
ROLLING_WINDOWS = config["features"]["rolling_windows"]
CYCLICAL_COLS = config["features"]["cyclical"]

## Load the cleaned dataset

In [13]:
df = load_raw_data(PROCESSED_PATH, TIMESTAMP_COL)
df = set_timestamp_index(df, TIMESTAMP_COL)
df.head()

,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event
timestamp,,,,,,,,,,
2020-01-01 00:00:00,1626.458412,29.264584,78.596059,2.010490,4.722857,283.256083,978.313852,489.264274,23.840111,1
2020-01-01 01:00:00,1593.917861,28.939179,78.981324,2.017491,3.804590,229.819735,935.434256,505.759101,24.308850,0
2020-01-01 02:00:00,1457.196911,27.571969,77.364594,1.972281,5.686889,263.726759,965.089335,491.523802,30.762810,0
2020-01-01 03:00:00,1302.073833,26.020738,79.678678,1.615092,5.024458,250.467925,988.898933,495.765616,26.104404,1
2020-01-01 04:00:00,1349.054422,26.490544,80.105671,2.280944,2.721659,255.340872,1032.194114,497.929332,22.300232,0


## Calendar features

In [14]:
df = add_calendar_features(df)
df.head()

,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event,hour,day_of_week,month,is_weekend
timestamp,,,,,,,,,,,,,,
2020-01-01 00:00:00,1626.458412,29.264584,78.596059,2.010490,4.722857,283.256083,978.313852,489.264274,23.840111,1,0,2,1,0
2020-01-01 01:00:00,1593.917861,28.939179,78.981324,2.017491,3.804590,229.819735,935.434256,505.759101,24.308850,0,1,2,1,0
2020-01-01 02:00:00,1457.196911,27.571969,77.364594,1.972281,5.686889,263.726759,965.089335,491.523802,30.762810,0,2,2,1,0
2020-01-01 03:00:00,1302.073833,26.020738,79.678678,1.615092,5.024458,250.467925,988.898933,495.765616,26.104404,1,3,2,1,0
2020-01-01 04:00:00,1349.054422,26.490544,80.105671,2.280944,2.721659,255.340872,1032.194114,497.929332,22.300232,0,4,2,1,0


## Lag features

Past demand values only (`shift`), so no future information leaks into a given row's features.

In [15]:
df = add_lag_features(df, TARGET_COL, LAGS)
df[[f"{TARGET_COL}_lag_{lag}" for lag in LAGS]].head()

,demand_lag_1,demand_lag_2,demand_lag_6,demand_lag_24,demand_lag_48,demand_lag_168
timestamp,,,,,,
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,1626.458412,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,1593.917861,1626.458412,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,1457.196911,1593.917861,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,1302.073833,1457.196911,NaN,NaN,NaN,NaN


## Rolling features

Computed on the lag-1 shifted series, so each row's rolling mean/std only reflects observations strictly before it.

In [16]:
df = add_rolling_features(df, TARGET_COL, ROLLING_WINDOWS)
df[[f"{TARGET_COL}_roll_mean_{w}" for w in ROLLING_WINDOWS]].head()

,demand_roll_mean_3,demand_roll_mean_6,demand_roll_mean_12,demand_roll_mean_24,demand_roll_mean_168
timestamp,,,,,
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,1559.191061,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,1451.062868,NaN,NaN,NaN,NaN


## Weather interaction feature

In [17]:
if "temperature" in df.columns and "humidity" in df.columns:
    df["temp_humidity_interaction"] = df["temperature"] * df["humidity"]
df.head()

,demand,temperature,humidity,wind_speed,rainfall,solar_irradiance,gdp,per_capita_energy_use,electricity_price,public_event,...,demand_roll_std_3,demand_roll_mean_6,demand_roll_std_6,demand_roll_mean_12,demand_roll_std_12,demand_roll_mean_24,demand_roll_std_24,demand_roll_mean_168,demand_roll_std_168,temp_humidity_interaction
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,1626.458412,29.264584,78.596059,2.010490,4.722857,283.256083,978.313852,489.264274,23.840111,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2300.080986
2020-01-01 01:00:00,1593.917861,28.939179,78.981324,2.017491,3.804590,229.819735,935.434256,505.759101,24.308850,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2285.654645
2020-01-01 02:00:00,1457.196911,27.571969,77.364594,1.972281,5.686889,263.726759,965.089335,491.523802,30.762810,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2133.094201
2020-01-01 03:00:00,1302.073833,26.020738,79.678678,1.615092,5.024458,250.467925,988.898933,495.765616,26.104404,1,...,89.815516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2073.298025
2020-01-01 04:00:00,1349.054422,26.490544,80.105671,2.280944,2.721659,255.340872,1032.194114,497.929332,22.300232,0,...,146.018677,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2122.042821


## Cyclical encoding

Sine/cosine encoding lets models see `hour=23` and `hour=0` as adjacent instead of far apart.

In [18]:
CYCLICAL_PERIODS = {"hour": 24, "day_of_week": 7, "month": 12}

for col in CYCLICAL_COLS:
    df = add_cyclical_features(df, col, CYCLICAL_PERIODS[col])

df[[f"{col}_{trig}" for col in CYCLICAL_COLS for trig in ("sin", "cos")]].head()

,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
timestamp,,,,,,
2020-01-01 00:00:00,0.000000,1.000000,0.974928,-0.222521,0.5,0.866025
2020-01-01 01:00:00,0.258819,0.965926,0.974928,-0.222521,0.5,0.866025
2020-01-01 02:00:00,0.500000,0.866025,0.974928,-0.222521,0.5,0.866025
2020-01-01 03:00:00,0.707107,0.707107,0.974928,-0.222521,0.5,0.866025
2020-01-01 04:00:00,0.866025,0.500000,0.974928,-0.222521,0.5,0.866025


## Drop warm-up rows

The longest lag/rolling window (168 hours = 1 week) leaves `NaN`s at the start of the series; these rows can't be used for training.

In [19]:
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} warm-up rows with incomplete lag/rolling history")
print(f"Final feature set: {df.shape[0]} rows, {df.shape[1]} columns")

Dropped 168 warm-up rows with incomplete lag/rolling history
Final feature set: 47304 rows, 37 columns


## Save the engineered feature set

In [20]:
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
df.reset_index().to_csv(FEATURES_PATH, index=False)
print(f"Saved {len(df)} rows and {df.shape[1]} columns to {FEATURES_PATH}")

Saved 47304 rows and 37 columns to /kaggle/working/data/processed/features.csv
